# exp-011: Stage 2 Reranker — Gemini RankZephyr 패턴

- **목적:** Top-100 후보를 Gemini API가 listwise rerank. 5관점별 system prompt 1줄 변경만으로 가중치 분기 효과 달성.
- **차별성 축:** ④ + ⑤ End-to-End LLM
- **데이터:** `search_results_top100.csv` (Top-100 후보) + 5관점 라벨
- **모델:** Gemini API (RankZephyr 패턴, listwise prompt)
- **메트릭:** 5관점 × NDCG@10 / MAP / Hit@1, vs exp-006 best 비교
- **비용 추정:** 10 users × 5 관점 × Top-100 rerank ≈ 50 API calls × 5-10초 ≈ $1-3
- **관련 위키:** RankZephyr, dual-encoder-진화-실험계획 exp-011
- **작성일:** 2026-05-18

## ⚠️ 실행 보류 사유

exp-005 Gemini 호출 충돌 우려. exp-005 종료 후 → exp-007 → exp-011 순서.

## Listwise Rerank Prompt 예시

```
당신은 한국어 채용 매칭 reranker. 아래 User와 100개 JD 후보 중
관점 {perspective}({label})에서 가장 관련 높은 순서로 1-10위 ID를 출력하세요.
[A=직무, B=자소서, C=스킬, D=산업/근무지, E=균형]
User: {user_profile_summary}
JD 후보 (id - title - role - industry):
[1] ... 
응답: JSON [{"rank": 1, "jd_id": ...}, ...]
```

In [ ]:
# ⚠️ 실행 보류 — exp-005 Gemini 충돌
import pandas as pd
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv('output/.env')

DATA = Path('raw/data/gemini_profile_outputs')
OUT_DIR = Path('raw/experiments/exp-011-gemini-reranker')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Top-100 후보 로드
top100 = pd.read_csv(DATA / 'search_results_top100.csv', encoding='utf-8-sig')
top100.columns = [c.lstrip('\ufeff') for c in top100.columns]
print(f'search_results_top100: {len(top100)} rows')
print(f'unique users: {top100["userId"].nunique() if "userId" in top100.columns else "check column"}')
print(top100.columns.tolist()[:10])

In [ ]:
# (실행 시) 5관점별 listwise rerank → NDCG@10, MAP, Hit@1
# vs exp-006 best 가중치 비교
print('⏸️ 실행 보류 중')